## Step 3: Segment-Level Mortality Experience Analysis

In this section, we analyze mortality experience across key segments such as age and gender.

Rather than relying on a single portfolio-level A/E ratio, actuarial experience studies evaluate where actual mortality deviates from expected assumptions. This helps identify patterns and supports assumption refinement.

We calculate Actual vs Expected (A/E) ratios across:
- Age groups
- Gender

This provides deeper insight into mortality risk drivers within the portfolio.

In [13]:
# Load clean table and generate data

import numpy as np
import pandas as pd
mortality_table = pd.read_csv("../data/processed/mortality_table.csv")
mortality_table.head()
portfolio = pd.read_csv("../data/processed/portfolio.csv")

In [25]:
portfolio["age_group"] = pd.cut(
    portfolio["age"],
    bins=[20, 30, 40, 50, 60, 70, 80],
    labels=["20-30", "30-40", "40-50", "50-60", "60-70", "70-80"])

### A/E Analysis by Age Group

In [26]:
age_analysis = portfolio.groupby("age_group").agg(
    expected_deaths=("expected_death_probability", "sum"),
    actual_deaths=("actual_death", "sum"))
age_analysis["ae_ratio"] = (
    age_analysis["actual_deaths"] / age_analysis["expected_deaths"])
age_analysis

/tmp/ipykernel_1425/2833597919.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_analysis = portfolio.groupby("age_group").agg(


,expected_deaths,actual_deaths,ae_ratio
age_group,,,
20-30,0.97028,2,2.061261
30-40,1.49630,3,2.004946
40-50,2.79420,4,1.431537
50-60,6.28464,8,1.272945
60-70,15.92272,18,1.130460
70-80,44.94132,44,0.979054


### A/E Analysis by Gender

In [27]:
gender_analysis = portfolio.groupby("sex").agg(
    expected_deaths=("expected_death_probability", "sum"),
    actual_deaths=("actual_death", "sum"))
gender_analysis["ae_ratio"] = (
    gender_analysis["actual_deaths"] / gender_analysis["expected_deaths"])
gender_analysis

,expected_deaths,actual_deaths,ae_ratio
sex,,,
female,29.05934,29,0.997958
male,43.42077,50,1.151523


### A/E Analysis by Age and Gender

In [28]:
age_gender_analysis = portfolio.groupby(["age_group", "sex"]).agg(
    expected_deaths=("expected_death_probability", "sum"),
    actual_deaths=("actual_death", "sum"))
age_gender_analysis["ae_ratio"] = (
    age_gender_analysis["actual_deaths"] /
    age_gender_analysis["expected_deaths"])
age_gender_analysis

/tmp/ipykernel_1425/1505365329.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_gender_analysis = portfolio.groupby(["age_group", "sex"]).agg(


expected_deaths  actual_deaths  ae_ratio
age_group sex                                             
20-30     female          0.29397              0  0.000000
          male            0.67631              2  2.957224
30-40     female          0.52892              1  1.890645
          male            0.96738              2  2.067440
40-50     female          0.99029              1  1.009805
          male            1.80391              3  1.663054
50-60     female          2.41379              1  0.414286
          male            3.87085              7  1.808388
60-70     female          6.10657              7  1.146306
          male            9.81615             11  1.120602
70-80     female         18.70260             19  1.015902
          male           26.23872             25  0.952790

In [29]:
age_analysis.describe()

,expected_deaths,actual_deaths,ae_ratio
count,6.000000,6.000000,6.000000
mean,12.068243,13.166667,1.480034
std,17.028710,16.203909,0.454264
min,0.970280,2.000000,0.979054
25%,1.820775,3.250000,1.166081
50%,4.539420,6.000000,1.352241
75%,13.513200,15.500000,1.861593
max,44.941320,44.000000,2.061261


In [30]:
age_analysis.to_csv("../data/processed/age_analysis.csv")
gender_analysis.to_csv("../data/processed/gender_analysis.csv")

### Key Insights

- Overall mortality experience is slightly higher than expected, with an A/E ratio of ~1.09, indicating a modest underestimation of mortality risk.

- Male policyholders exhibit higher-than-expected mortality (A/E > 1.15), while female mortality aligns closely with expected assumptions.

- Mortality experience increases with age, with the 50–70 age groups consistently showing A/E ratios above 1.1, suggesting potential underestimation of mortality in middle-to-older age segments.

- Younger age groups show volatile A/E ratios due to low exposure and limited expected deaths, which is typical in experience studies.

### Interpretation

These results suggest that mortality assumptions may require adjustment for specific segments, particularly males and older age groups, to improve pricing accuracy and risk management.

## Step 4: Financial Impact Analysis
In this section, we extend the mortality experience study by analyzing the financial impact of mortality outcomes.

We compare expected and actual claims to evaluate how deviations in mortality affect insurer liabilities and portfolio risk.

In [31]:
portfolio.columns

Index(['policy_id', 'age', 'sex', 'policy_duration', 'sum_insured',
       'survivors_lx', 'deaths_dx', 'mortality_rate_qx',
       'central_death_rate_mx', 'survival_probability_px',
       'person_years_lived_Lx', 'total_future_years_Tx', 'life_expectancy_ex',
       'std_error_life_expectancy', 'expected_death_probability',
       'actual_death', 'claim_amount', 'age_group', 'expected_claim'],
      dtype='object')

In [33]:
portfolio["claim_amount"] = portfolio["actual_death"] * portfolio["sum_insured"]
portfolio["expected_claim"] = (
    portfolio["expected_death_probability"] * portfolio["sum_insured"])

In [34]:
total_expected_claims = portfolio["expected_claim"].sum()
total_actual_claims = portfolio["claim_amount"].sum()

claim_ae_ratio = total_actual_claims / total_expected_claims

print("Expected Claims:", total_expected_claims)
print("Actual Claims:", total_actual_claims)
print("Claim A/E Ratio:", claim_ae_ratio)

Expected Claims: 37300619.378800005
Actual Claims: 43699798
Claim A/E Ratio: 1.171556899798747


In [35]:
claims_by_age = portfolio.groupby("age_group").agg(
    expected_claims=("expected_claim", "sum"),
    actual_claims=("claim_amount", "sum"))
claims_by_age["ae_ratio"] = (
    claims_by_age["actual_claims"] / claims_by_age["expected_claims"])
claims_by_age

/tmp/ipykernel_1425/2245632035.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  claims_by_age = portfolio.groupby("age_group").agg(


,expected_claims,actual_claims,ae_ratio
age_group,,,
20-30,5.085137e+05,868454,1.707828
30-40,7.908009e+05,2294086,2.900965
40-50,1.456596e+06,2675460,1.836789
50-60,3.321645e+06,4245591,1.278159
60-70,8.398334e+06,10306538,1.227212
70-80,2.278892e+07,23309669,1.022851


In [36]:
claims_by_gender = portfolio.groupby("sex").agg(
    expected_claims=("expected_claim", "sum"),
    actual_claims=("claim_amount", "sum"))
claims_by_gender["ae_ratio"] = (
    claims_by_gender["actual_claims"] / claims_by_gender["expected_claims"])
claims_by_gender

,expected_claims,actual_claims,ae_ratio
sex,,,
female,1.482108e+07,16363421,1.104064
male,2.247954e+07,27336377,1.216056


### Financial Insights

- Actual claims exceeded expected claims, consistent with the A/E mortality ratio above 1.0, indicating higher-than-anticipated insurer payouts.

- Higher claim costs are concentrated in older age groups, reflecting both increased mortality and higher exposure levels.

- Male policyholders contribute disproportionately to excess claims due to higher-than-expected mortality.

### Interpretation

The deviation between expected and actual claims highlights the financial impact of inaccurate mortality assumptions. Adjusting mortality bases for higher-risk segments could improve pricing accuracy and capital planning.